# 0️⃣6️⃣ - How to transfer files using SCP/SFTP
![Persona](https://img.shields.io/badge/%F0%9F%91%A4%20Persona-%F0%9F%A4%96%20Network%20Automation%20Developer-lightgrey) ![Difficulty](https://img.shields.io/badge/%F0%9F%9A%A6%20Difficulty-Intermediate-yellow) ![RADKit version](https://img.shields.io/badge/RADKit-1.9.6-blue?logo=cisco&logoColor=white) ![Python version](https://img.shields.io/badge/Python-3.12%2B-yellow?logo=python&logoColor=white)

> In this playbook we will explore how to open interactive terminal sessions and transfer files to and from devices using SCP/SFTP.

### 🛠️ Before You Begin

If you haven't done it yet, follow the instructions [in this file](../SETUP.md) to setup your environment and run this playbook.

---

### 📋 What You'll Learn

| # | Topic |
|---|---|
| 1 | Download files from devices using SFTP/SCP |
| 2 | Upload files to devices using SFTP/SCP |
| 3 | Implement protocol failover for reliability |

---

RADKit comes bundled with SCP/SFTP utilities for transfering and uploading files. This is specially useful when downloading file dumps for troubleshooting purposes.

## ↔️ Method 1: Download Files Using SFTP/SCP

**Best for:** Backing up configs, collecting logs, extracting device dumps, or retrieving any files from your devices.

**How it works:**
1. Create a client and authenticate via SSO
2. Retrieve the device from your service inventory
3. Execute a command on the device to prepare the file (optional)
4. Call `sftp_download_to_file()` or `scp_download_to_file()`
5. Display a progress bar and wait for the transfer to complete

**What you need:**
- Valid CCO credentials
- Service ID and device name
- Source file path on the device (remote path)
- Destination file path on your local machine (local path)

### 1.1 Backup Configuration and Download

---

In [ ]:
from radkit_client import Client

user_id = input("👤 Enter your CCO user id: ")
service_id = input("🛠️ Enter the service id you want to connect to: ")

with Client.create() as client:
    client.sso_login(user_id)
    service = client.service_cloud(service_id).wait()
    my_device = service.inventory['p0-2e']
    
    print("📁 Backing up startup-config to flash:/my-backup.cfg ...\n")
    print(my_device.exec("copy startup-config flash:/my-backup.cfg").wait().data)
    
    print("\n📁 Verifying the file exists on the device with `dir flash:/my-backup.cfg`...\n")
    print(my_device.exec("dir flash:/my-backup.cfg").wait().data)
    print("\n")
    
    # Trying SFTP first, then SCP if it fails.
    transfer_ok = False
    last_error = None
    for protocol in ("sftp", "scp"):
        try:
            if protocol == "sftp":
                req = my_device.sftp_download_to_file(remote_path="flash:/my-backup.cfg", local_path="my-backup.cfg")
            else:
                req = my_device.scp_download_to_file(remote_path="flash:/my-backup.cfg", local_path="my-backup.cfg")
                
            # A progress bar will be shown
            req.show_progress()
            # The session will be closed once the download is complete, so we can wait for that to know when the transfer is done.
            req.wait_closed()
            
            print(f"✅ Download completed with {protocol.upper()} from 'flash:/my-backup.cfg' -> 'my-backup.cfg'")
            transfer_ok = True
            break
        except Exception as exc:
            last_error = exc
            print(f"❌ {protocol.upper()} failed for 'flash:/my-backup.cfg': {exc}")
        if transfer_ok:
            break

    if not transfer_ok:
        raise RuntimeError(f"❌ Download failed with all path/protocol combinations. Last error: {last_error}")


A browser window was opened to continue the authentication process. Please follow the instructions there.

Authentication result received.
📁 Backing up startup-config to flash:/my-backup.cfg ...

p0-2E#copy startup-config flash:/my-backup.cfg
88864 bytes copied in 0.025 secs (3554560 bytes/sec)
p0-2E#

📁 Verifying the file exists on the device with `dir flash:/my-backup.cfg`...

p0-2E#dir flash:/my-backup.cfg
Directory of flash:/my-backup.cfg
 
229472  -rw-            88864   Apr 1 2026 14:14:42 +00:00  my-backup.cfg
 
11353194496 bytes total (5184876544 bytes free)
p0-2E#


❌ SFTP failed for 'flash:/my-backup.cfg': Performing action failed: 0 bytes read on a total of 4 expected bytes
my-backup.cfg   0.0% [>                             ]   0/88864   eta [?:??:??]






 my-backup.cfg 100.0% [===============================>] 88864/88864 eta [00:00]
✅ Download completed with SCP from 'flash:/my-backup.cfg' -> 'my-backup.cfg'


## ↔️ Method 2: Upload Files Using SFTP/SCP

**Best for:** Pushing configurations, scripts, firmware files, or any local content to your devices.

**How it works:**
1. Create a client and authenticate via SSO
2. Retrieve the device from your service inventory
3. Call `sftp_upload_from_file()` or `scp_upload_from_file()`
4. Display a progress bar and wait for the transfer to complete
5. Execute a verification command on the device (optional)

**What you need:**
- Valid CCO credentials
- Service ID and device name
- Source file path on your local machine (local path)
- Destination file path on the device (remote path)

### 2.1 Upload Configuration File and Verify

---

In [20]:
from radkit_client import Client

user_id = input("👤 Enter your CCO user id: ")
service_id = input("🛠️ Enter the service id you want to connect to: ")

with Client.create() as client:
    client.sso_login(user_id)
    service = client.service_cloud(service_id).wait()
    my_device = service.inventory['p0-2e']
    
    # Trying SFTP first, then SCP if it fails.
    transfer_ok = False
    last_error = None
    for protocol in ("sftp", "scp"):
        try:
            if protocol == "sftp":
                req = my_device.sftp_upload_from_file(remote_path="flash:/my-backup_.cfg", local_path="my-backup.cfg")
            else:
                req = my_device.scp_upload_from_file(remote_path="flash:/my-backup_.cfg", local_path="my-backup.cfg")
                
            # A progress bar will be shown
            req.show_progress()
            # The session will be closed once the upload is complete, so we can wait for that to know when the transfer is done.
            req.wait_closed()
            
            print(f"✅ Upload completed with {protocol.upper()} from 'my-backup.cfg' -> 'flash:/my-backup_.cfg'")
            transfer_ok = True
            break
        except Exception as exc:
            last_error = exc
            print(f"❌ {protocol.upper()} failed for 'flash:/my-backup_.cfg': {exc}")
        if transfer_ok:
            break

    if not transfer_ok:
        raise RuntimeError(f"❌ Upload failed with all path/protocol combinations. Last error: {last_error}")
    else:
        print("\n📁 Verifying the file exists on the device with `dir flash:/my-backup_.cfg`...\n")
        print(my_device.exec("dir flash:/my-backup_.cfg").wait().data)
        print("\n")


A browser window was opened to continue the authentication process. Please follow the instructions there.

Authentication result received.
❌ SFTP failed for 'flash:/my-backup_.cfg': Performing action failed: 0 bytes read on a total of 4 expected bytes
my-backup.cfg   0.0% [>                             ]   0/88864   eta [?:??:??]






 my-backup.cfg 100.0% [===============================>] 88864/88864 eta [00:00]
✅ Upload completed with SCP from 'my-backup.cfg' -> 'flash:/my-backup_.cfg'

📁 Verifying the file exists on the device with `dir flash:/my-backup_.cfg`...

p0-2E#dir flash:/my-backup_.cfg
Directory of flash:/my-backup_.cfg
 
229475  -rw-            88864   Apr 1 2026 14:22:20 +00:00  my-backup_.cfg
 
11353194496 bytes total (5184782336 bytes free)
p0-2E#




## 🔀 Which Method Should I Use?

| Dimension | Method 1: Download | Method 2: Upload |
|-----------|-------------------|-----------------|
| **Use case** | Retrieve files from devices | Push files to devices |
| **Example** | Backup configs, collect logs | Deploy scripts, upload firmware |
| **Protocol** | SFTP with SCP failover | SFTP with SCP fallback |
| **Progress visibility** | Yes, via `show_progress()` | Yes, via `show_progress()` |
| **Verify after** | Supported (shown in example) | Supported (shown in example) |